# Notebook 12 - EU relevance text fixture

Converts one long report and 14 short briefing notes from PDF to clean markdown. The output is a text fixture only:
this notebook computes no distance, no embedding-based reading of a note against the report and no relevance
judgement. Labels are assigned later, elsewhere.

- **Report** - European Commission, *State of the Digital Decade 2024*, Annex 1 (76 pages, CC BY 4.0)
- **Notes** - 14 European Parliamentary Research Service "At a glance" notes on related digital-policy subjects
- **Input** - the PDFs in `data/external/` (provenance in `data/external/eu-digital-decade-relevance_origin.md`)
- **Output** - `data/processed/relevance-fixture-eu/`: `report.md`, `briefs/<PE>-<slug>.md`, `sections.json`, `README.md`

Format rule that drives the conversion: the markdown reader of the package makes one statement per heading, list item and
table row, cuts paragraph text into sentences, cuts at every line break and removes the first `# ` title line. A paragraph
therefore has to sit on one line; a wrapped paragraph would be cut at each line end.

Stages:

1. **Convert the report** - keep text characters by font and size, build lines, classify headings, join paragraphs
2. **Convert the notes** - same helpers, size rules of the EPRS layout
3. **Write and read back** - write the files, count statements with the package reader, build the heading tree
4. **Check** - residue statistics, sample read-through, reading order, word accounting


## Environment

Data-prep notebook: no model needs a GPU. The only model is the `sat-3l-sm` statement segmenter (CPU), loaded offline.
`DOCDISTANCE_HOME` is set to the repository root before the package is imported.


In [1]:
import os

# the package reads DOCDISTANCE_HOME for its data paths; no model needs a GPU in this notebook
os.environ["DOCDISTANCE_HOME"] = "/home/lab/workspace/learning/projects/docdistance"
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["CI"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import warnings
warnings.filterwarnings("ignore")
print("environment set: CPU only, hub offline, DOCDISTANCE_HOME =", os.environ["DOCDISTANCE_HOME"])


environment set: CPU only, hub offline, DOCDISTANCE_HOME = /home/lab/workspace/learning/projects/docdistance


## Imports


In [2]:
# stdlib
import collections
import json
import random
import re
import statistics
from pathlib import Path

# third party
import pdfplumber
from rich.console import Console
from rich.table import Table

# local - the markdown reader of the package, used for segmentation only (no encoder is loaded)
from docdistance.encoders import Segmenter, split_markdown
from docdistance.pipeline import _strip_titles

console = Console(width=140)
print("imports ready")


imports ready


## Reproducibility

The only random choice is the sample of 25 report statements for the read-through.


In [3]:
SEED = 0
print(f"SEED={SEED}")


SEED=0


## Configuration

Paths, the page range of the report, the 14 notes (PE number to file slug) and the layout constants in one place.


In [4]:
ROOT = Path(os.environ["DOCDISTANCE_HOME"])
EXTERNAL = ROOT / "data/external"
OUT_DIR = ROOT / "data/processed/relevance-fixture-eu"
BRIEFS_DIR = OUT_DIR / "briefs"
for _d in (OUT_DIR, BRIEFS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

REPORT_PDF = EXTERNAL / "eu-digital-decade-2024-annex1.pdf"
REPORT_TITLE = ("State of the Digital Decade 2024, Annex 1 - Competitiveness and sovereignty, people, "
                "smart greening, policy coherence and synergies")

# report pages: 1 cover, 2 legal notice, 3 blank page, 4 to 75 content, 76 back cover with the imprint
REPORT_FIRST_PAGE, REPORT_LAST_PAGE = 4, 75

# EPRS "At a glance" notes: PE number -> slug of the file name
BRIEFS = {
    "760421": "gigabit-infrastructure-act",
    "751386": "eu-chips-act",
    "759620": "european-media-freedom-act",
    "747926": "ai-act-parliament-position",
    "772906": "ai-act-implementation-timeline",
    "779252": "cybersecurity-act-review",
    "760355": "foreign-interference-preparedness",
    "779193": "simplifying-digital-laws",
    "739206": "european-year-of-skills",
    "739350": "online-age-verification",
    "754642": "european-health-data-space",
    "760392": "ai-investment-indicators",
    "759583": "strategic-technologies-platform",
    "760391": "gdpr-cross-border-procedural-rules",
}

# a gap between two lines larger than this multiple of the normal line pitch starts a new paragraph
PARAGRAPH_GAP = 1.12
# markdown headings stop at six hashes; the package reader does not read a seventh as a heading
MAX_HEADING_LEVEL = 6
BULLET = "•"
# word pieces that keep their hyphen when a line break splits them and no other evidence exists
HYPHEN_PREFIXES = ("non", "self", "cross", "anti", "semi", "ex")
# a split verified against the PDF where the vocabulary rule picks the wrong form
HYPHEN_VERIFIED = {("wide", "bandgap")}
print(f"report: {REPORT_PDF.name} (exists={REPORT_PDF.exists()}); briefs: {len(BRIEFS)}; output: {OUT_DIR}")


report: eu-digital-decade-2024-annex1.pdf (exists=True); briefs: 14; output: /home/lab/workspace/learning/projects/docdistance/data/processed/relevance-fixture-eu


## Stage 1 - Shared conversion helpers

Both document types go through the same steps: characters are filtered by font and size before lines are formed (so a chart
label at the same height as a body line does not enter the line), lines are grouped into blocks by the vertical gap, a block
cut by a page break, figure or box is joined again when the next piece continues the sentence, and a word split by a line
break is joined by a vocabulary rule. The vocabulary comes from all fifteen documents; where neither form of a split word
occurs, the shape of the two pieces decides and the case is listed.


In [5]:
WHOLE_SHARE = 0.95
TERMINAL_RE = re.compile(r"[.!?:;]$")
WORD_RE = re.compile(r"[A-Za-z]+(?:-[A-Za-z]+)*")


def char_style(char: dict) -> tuple[str, float]:
    """Style word and size of a PDF character, from its font name and size.

    Args:
        char: a pdfplumber character record.

    Returns:
        ``(style, size)``, for example ``("Bold", 12.0)``; ``It`` is named ``Italic``.
    """
    suffix = char["fontname"].split("+")[-1].rsplit("-", 1)[-1]
    return {"It": "Italic"}.get(suffix, suffix), round(char["size"], 1)


def read_lines(page, keep, tidy_punctuation: bool = False) -> list[dict]:
    """Lines of one page, built from the characters that ``keep`` accepts.

    Characters are removed before the lines are formed, so text that shares a line with a removed
    chart label or caption comes out clean.

    Args:
        page: a pdfplumber page.
        keep: predicate on a character record.
        tidy_punctuation: remove the space that a removed footnote marker leaves before a mark.

    Returns:
        Line records in top-to-bottom order: ``text``, ``x0``, ``top``, ``style`` and ``size`` of
        the dominant character style, and ``whole`` (95 % or more of the characters share it).
    """
    kept = page.filter(lambda obj: obj["object_type"] != "char" or keep(obj))
    lines = []
    for raw in kept.extract_text_lines(return_chars=True):
        chars = [c for c in raw["chars"] if c["text"].strip()]
        if not chars:
            continue
        counts = collections.Counter(char_style(c) for c in chars)
        (style, size), share = counts.most_common(1)[0]
        text = raw["text"].strip()
        if tidy_punctuation:
            text = re.sub(r"\s+([.,;:!?)])", r"\1", text)
        lines.append({"text": text, "x0": raw["x0"], "top": raw["top"], "style": style,
                      "size": size, "whole": share / len(chars) >= WHOLE_SHARE})
    return lines


def line_pitch_ratios(page_lines: list[tuple[int, list[dict]]], classify) -> dict[float, float]:
    """Median line pitch of body text per font size, as a multiple of the size.

    The pitch is measured between consecutive text lines of one page. Most gaps lie inside a
    paragraph, so the median is the normal line pitch of the document; it differs between documents
    (1.14 to 1.40 of the size) and a paragraph gap is only a little larger in some of them.
    A size with fewer than five gaps takes the median over all sizes.
    """
    by_size = collections.defaultdict(list)
    for _, lines in page_lines:
        kinds = classify(lines)
        for i in range(1, len(lines)):
            if kinds[i] == kinds[i - 1] == "text" and lines[i]["size"] == lines[i - 1]["size"]:
                by_size[lines[i]["size"]].append((lines[i]["top"] - lines[i - 1]["top"]) / lines[i]["size"])
    pooled = statistics.median(r for ratios in by_size.values() for r in ratios)
    return collections.defaultdict(lambda: pooled, {
        size: statistics.median(ratios) for size, ratios in by_size.items() if len(ratios) >= 5})


def build_blocks(page_lines: list[tuple[int, list[dict]]], classify, merge_kinds: set[str]):
    """Group the lines of a document into paragraph, list-item and heading blocks.

    A new paragraph starts at a gap larger than ``PARAGRAPH_GAP`` times the median line pitch of
    that font size, at a bullet, at a change of font size and after a heading. A paragraph cut by
    a page break, a figure or a box is joined again when the next piece starts with a lower-case
    letter (a page break joins also when the cut paragraph has no closing punctuation).

    Args:
        page_lines: ``(page number, lines)`` in reading order.
        classify: function from the lines of a page to one kind per line (``text`` or a heading kind).
        merge_kinds: heading kinds whose consecutive lines form one wrapped heading.

    Returns:
        ``(blocks, breaks)``. A block has ``kind``, ``lines``, ``size`` and ``page``; ``breaks``
        lists each cut paragraph with the decision taken, for inspection.
    """
    ratios = line_pitch_ratios(page_lines, classify)
    blocks, breaks = [], []
    for page_no, lines in page_lines:
        kinds = classify(lines)
        for i, ln in enumerate(lines):
            gap = ln["top"] - lines[i - 1]["top"] if i else None
            kind, last = kinds[i], (blocks[-1] if blocks else None)
            if kind != "text":
                wrapped = (last is not None and last["kind"] == kind and kind in merge_kinds
                           and gap is not None and gap <= 1.6 * ln["size"]
                           and not re.match(r"\d+(\.\d+)*\.?\s", ln["text"]))
                if wrapped:
                    last["lines"].append(ln["text"])
                else:
                    blocks.append({"kind": kind, "lines": [ln["text"]], "size": ln["size"], "page": page_no})
                continue
            start = (last is None or last["kind"] != "text" or ln["text"].startswith(BULLET)
                     or last["size"] != ln["size"])
            cut_by_page = gap is None
            cut_by_gap = gap is not None and gap > PARAGRAPH_GAP * ratios[ln["size"]] * ln["size"]
            if not start and (cut_by_page or cut_by_gap):
                closed = TERMINAL_RE.search(last["lines"][-1]) is not None
                lower = ln["text"][:1].islower()
                joined = (lower or not closed) if cut_by_page else (lower and not closed)
                breaks.append((page_no, "page" if cut_by_page else "gap", joined,
                               last["lines"][-1][-40:], ln["text"][:40]))
                start = not joined
            if start:
                blocks.append({"kind": "text", "lines": [ln["text"]], "size": ln["size"], "page": page_no})
            else:
                last["lines"].append(ln["text"])
    return blocks, breaks


def build_vocabulary(blocks: list[dict]) -> collections.Counter:
    """Lower-case word counts of all blocks, leaving out the two halves of a split word."""
    vocabulary = collections.Counter()
    for block in blocks:
        lines = block["lines"]
        for i, line in enumerate(lines):
            if line.endswith("-") and i + 1 < len(lines):
                line = line.rsplit(" ", 1)[0]
            if i and lines[i - 1].endswith("-"):
                line = line.split(" ", 1)[1] if " " in line else ""
            vocabulary.update(w.lower() for w in WORD_RE.findall(line))
    return vocabulary


def hyphen_verdict(head: str, tail: str, vocabulary: collections.Counter) -> tuple[str, str]:
    """Decide whether a word split at a line break keeps its hyphen.

    Returns:
        ``("hyphen" | "glue", reason)``; the reason is ``seen`` when the vocabulary holds one of
        the two forms, ``verified`` for a split checked against the PDF, and ``fallback`` when
        only the shape of the two pieces decides.
    """
    glued, hyphened = vocabulary[head + tail], vocabulary[head + "-" + tail]
    if (head, tail) in HYPHEN_VERIFIED:
        return "hyphen", "verified"
    if hyphened > glued:
        return "hyphen", "seen"
    if glued:
        return "glue", "seen"
    pieces_are_words = vocabulary[head] >= 1 and vocabulary[tail] + vocabulary[tail + "s"] + vocabulary[tail.rstrip("s")] >= 1
    return ("hyphen" if pieces_are_words or head in HYPHEN_PREFIXES else "glue"), "fallback"


def join_lines(lines: list[str], vocabulary: collections.Counter, decisions: list) -> str:
    """Join the wrapped lines of one block into one line, repairing words split at a line break."""
    out = lines[0]
    for nxt in lines[1:]:
        if out.endswith("-") and out[-2:-1].isalpha() and re.match(r"[a-z]", nxt):
            head = out.rsplit(" ", 1)[-1][:-1].lower().split("-")[-1]
            tail = re.match(r"[a-z]+", nxt).group(0)
            verdict, reason = hyphen_verdict(head, tail, vocabulary)
            decisions.append((head, tail, verdict, reason))
            out = out[:-1] + ("-" if verdict == "hyphen" else "") + nxt
        elif out.endswith("-") and out[-2:-1].isalnum():
            out += nxt
        else:
            out += " " + nxt
    return out
print("helpers defined")


helpers defined


### Convert the annex

How the annex PDF is read (fonts found with `page.chars`):

- body text Regular and Bold 11 pt; bold runs inside a paragraph are emphasis, not headings
- chapter headings Bold 24 pt; section headings Bold 12 pt, numbered (`1.`, `1.1.`, `2.1.3.1`) or unnumbered
- recommendation and text boxes 10 pt, with a bold-italic box title and bold box labels
- removed by style: chart text (under 9 pt), figure captions (Medium Italic 11 pt), footnotes (Italic 8 pt), footnote markers
  (Italic 6.4 pt), page numbers (Medium 8 pt); cover, legal notice, blank page and imprint pages by page number


In [6]:
REPORT_TEXT_STYLES = ("Regular", "Bold", "BoldItalic", "Italic")
REPORT_MIN_TEXT_SIZE = 9.5
NUMBERED_HEADING_RE = re.compile(r"^(\d+(?:\.\d+)*)\.?\s+(\S.*)$")
LABEL_MAX_CHARS = 120
LABEL_MIN_GAP = 17


def is_report_text(char: dict) -> bool:
    """Body, heading and box text of the annex: upright or italic text of 9.5 pt and above.

    Chart labels (under 9 pt), figure captions (Medium Italic 11 pt), footnotes (Italic 8 pt),
    footnote markers (Italic 6.4 pt) and page numbers (Medium 8 pt) fall outside.
    """
    style, size = char_style(char)
    return style in REPORT_TEXT_STYLES and size >= REPORT_MIN_TEXT_SIZE


def classify_report_page(lines: list[dict]) -> list[str]:
    """One kind per line: head24, head12, boxtitle, boxlabel or text.

    A fully bold 24 pt line is a chapter heading and a fully bold 12 pt line a section heading.
    A run of fully bold 10 pt lines that stands alone between two block gaps is a box title when
    it contains ``Recommended`` and a box label otherwise.
    """
    kinds = []
    for ln in lines:
        style, size = ln["style"], ln["size"]
        if not ln["whole"]:
            kinds.append("text")
        elif style == "Bold" and size == 24.0:
            kinds.append("head24")
        elif style == "Bold" and size == 12.0:
            kinds.append("head12")
        elif style in ("Bold", "BoldItalic") and size == 10.0:
            kinds.append("bold10")
        else:
            kinds.append("text")
    i = 0
    while i < len(lines):
        if kinds[i] != "bold10":
            i += 1
            continue
        j = i
        while j + 1 < len(lines) and kinds[j + 1] == "bold10" and lines[j + 1]["top"] - lines[j]["top"] <= 16:
            j += 1
        before = None if i == 0 else lines[i]["top"] - lines[i - 1]["top"]
        after = None if j + 1 == len(lines) else lines[j + 1]["top"] - lines[j]["top"]
        text = " ".join(ln["text"] for ln in lines[i:j + 1])
        alone = (before is None or before >= LABEL_MIN_GAP) and (after is None or after >= LABEL_MIN_GAP)
        is_label = alone and len(text) <= LABEL_MAX_CHARS and not text.endswith(".")
        kind = ("boxtitle" if "Recommended" in text else "boxlabel") if is_label else "text"
        kinds[i:j + 1] = [kind] * (j - i + 1)
        i = j + 1
    return kinds


def convert_report_pdf(path: Path):
    """Blocks of the annex in reading order, the cut-paragraph log and the dropped styles."""
    dropped = collections.Counter()
    page_lines = []
    with pdfplumber.open(path) as pdf:
        for page_no in range(REPORT_FIRST_PAGE, REPORT_LAST_PAGE + 1):
            page = pdf.pages[page_no - 1]
            for char in page.chars:
                if not is_report_text(char) and char_style(char)[1] >= REPORT_MIN_TEXT_SIZE:
                    dropped[char_style(char)] += 1
            page_lines.append((page_no, read_lines(page, is_report_text, tidy_punctuation=True)))
    blocks, breaks = build_blocks(page_lines, classify_report_page, {"head24", "head12", "boxtitle", "boxlabel"})
    return blocks, breaks, dropped


def render_report(blocks: list[dict], vocabulary, decisions: list):
    """Markdown text of the annex and its heading records.

    Heading level follows the numbering depth (``1.`` is ``##``, ``1.1.`` is ``###``). An unnumbered
    section heading sits one level below the numbered heading before it, a box title one level
    below that, and a box label one level below the box title; no level passes six hashes.
    """
    parent, title_level = 2, None
    lines, heads = [f"# {REPORT_TITLE}"], []
    for block in blocks:
        text = join_lines(block["lines"], vocabulary, decisions)
        kind = block["kind"]
        if kind == "text":
            lines.append("- " + text[1:].strip() if text.startswith(BULLET) else text)
            continue
        numbered = NUMBERED_HEADING_RE.match(text) if kind in ("head24", "head12") else None
        if numbered:
            level, role = len(numbered.group(1).split(".")) + 1, "numbered"
            parent, title_level = level, None
        elif kind == "head24":
            level, parent, title_level, role = 2, 2, None, "chapter"
        elif kind == "head12":
            level, title_level, role = parent + 1, None, "subheading"
        elif kind == "boxtitle":
            level, role = min(parent + 1, MAX_HEADING_LEVEL), "boxtitle"
            title_level = level
        else:
            level, role = min((title_level or parent) + 1, MAX_HEADING_LEVEL), "boxlabel"
        heads.append({"level": level, "role": role, "text": text, "page": block["page"]})
        lines.append("#" * level + " " + text)
    return "\n\n".join(lines) + "\n", heads


report_blocks, report_breaks, report_dropped = convert_report_pdf(REPORT_PDF)
print(f"annex: {len(report_blocks)} blocks, {len(report_breaks)} cut paragraphs inspected")
print("kinds:", dict(collections.Counter(b["kind"] for b in report_blocks)))
print("styles of 9.5 pt and above dropped (characters):", dict(report_dropped))


annex: 438 blocks, 197 cut paragraphs inspected
kinds: {'head24': 6, 'text': 325, 'head12': 43, 'boxtitle': 24, 'boxlabel': 40}
styles of 9.5 pt and above dropped (characters): {('MediumItalic', 11.0): 1935, ('Medium', 10.9): 37}


### Convert the notes

The EPRS notes have one layout in different fonts: title 24 pt and above, section headings 14 pt, bold 12 pt sub-headings,
body 11 pt, lead paragraph and boxes about 10.5 pt, figure captions 12 pt regular, table cells and footnotes 9 pt, disclaimer 8 pt.
The EPRS footer is cut at its first line (`EPRS | European Parliamentary Research Service`) and everything below it on the page.
Two notes use a Wingdings arrow as list marker (`U+F0D8`), written as `- `.


In [7]:
NOTE_BODY_SIZE_RANGE = (10.0, 11.3)
NOTE_SECTION_SIZE_RANGE = (13.5, 14.5)
NOTE_TITLE_MIN_SIZE = 24.0
NOTE_SUBHEADING_BOLD_SIZE = 12.0
NOTE_BOX_TITLE_SIZE_RANGE = (10.0, 10.9)
SUPERSCRIPT_SIZE_RANGE = (6.0, 7.0)
FOOTER_ANCHOR = "EPRS | European Parliamentary Research Service"
PUA_BULLET = "\uf0d8"  # the Wingdings arrow that two notes use as list marker
SOURCE_LINE_RE = re.compile(r"^Sources?:")
SOURCE_SIZE_RANGE = (10.0, 10.1)
NOTE_MARKER_RE = re.compile(r"(?<=\w)\*+")


def in_range(value: float, bounds: tuple[float, float]) -> bool:
    return bounds[0] <= value <= bounds[1]


def is_note_text(char: dict) -> bool:
    """Body, heading and title text of a note by size and weight.

    Body and box text are 10.0 to 11.3 pt. A section heading is 14 pt regular, a sub-heading is
    12 pt bold, a title is 24 pt and above. Series labels (16 and 20 pt), figure captions (12 pt
    regular), table cells and footnotes (9 pt) and the disclaimer (8 pt and below) fall outside.
    """
    style, size = char_style(char)
    return (in_range(size, NOTE_BODY_SIZE_RANGE) or in_range(size, NOTE_SECTION_SIZE_RANGE)
            or size >= NOTE_TITLE_MIN_SIZE or (style == "Bold" and size == NOTE_SUBHEADING_BOLD_SIZE))


def char_key(char: dict) -> tuple:
    """Identity of a character on its page: position and glyph."""
    return char["x0"], char["top"], char["text"]


def source_line_keys(page, removed: list) -> set:
    """Characters of the figure source lines (``Source: ...``, 10 pt and below) of one page.

    The lines are read from the small-size characters alone, because a source line often stands
    at the same height as a line of body text in the other column.
    """
    small = page.filter(lambda o: o["object_type"] != "char" or in_range(char_style(o)[1], SOURCE_SIZE_RANGE))
    keys, in_source, previous = set(), False, None
    for raw in small.extract_text_lines(return_chars=True):
        gap = raw["top"] - previous["top"] if previous else None
        in_source = bool(SOURCE_LINE_RE.match(raw["text"])) or (
            in_source and gap is not None and gap <= 1.6 * 10.0 and raw["x0"] == previous["x0"])
        if in_source:
            keys |= {char_key(c) for c in raw["chars"]}
            removed.append(("source line", raw["text"]))
        previous = raw
    return keys


def note_char_filter(page, removed: list):
    """Predicate for one page: the text characters, superscripts touching them (``10^25``) and word spaces.

    Figure source lines are left out (see :func:`source_line_keys`).
    """
    body = [c for c in page.chars if is_note_text(c)]
    keys = {char_key(c) for c in body} - source_line_keys(page, removed)
    small = [c for c in page.chars if in_range(c["size"], SUPERSCRIPT_SIZE_RANGE)]
    grew = True
    while grew:  # a two-digit exponent: the second digit touches the first, not the body text
        grew = False
        for c in small:
            touching = any(abs(c["x0"] - b["x1"]) <= 1.5 and b["top"] - 4 <= c["top"] <= b["bottom"] for b in body)
            if char_key(c) not in keys and touching:
                keys.add(char_key(c))
                body.append(c)
                grew = True
    # some notes set a word space in 8 pt inside 11 pt text; a space that touches a kept character stays
    for c in page.chars:
        if c["text"].isspace() and char_key(c) not in keys and any(
                (abs(c["x0"] - b["x1"]) <= 1.5 or abs(c["x1"] - b["x0"]) <= 1.5) and abs(c["top"] - b["top"]) <= 6
                for b in body):
            keys.add(char_key(c))
    return lambda obj: char_key(obj) in keys


def classify_note_page(lines: list[dict]) -> list[str]:
    """One kind per line: title, h2 (section), h3 (sub-heading or box title) or text."""
    kinds = []
    for ln in lines:
        style, size = ln["style"], ln["size"]
        if size >= NOTE_TITLE_MIN_SIZE:
            kinds.append("title")
        elif not ln["whole"]:
            kinds.append("text")
        elif style != "Bold" and in_range(size, NOTE_SECTION_SIZE_RANGE):
            kinds.append("h2")
        elif style == "Bold" and (size == NOTE_SUBHEADING_BOLD_SIZE or in_range(size, NOTE_BOX_TITLE_SIZE_RANGE)):
            kinds.append("h3")
        else:
            kinds.append("text")
    return kinds


def drop_footer(lines: list[dict], removed: list) -> list[dict]:
    """Remove the EPRS footer: the anchor line and every line below it."""
    anchor = next((i for i, ln in enumerate(lines) if ln["text"].startswith(FOOTER_ANCHOR)), len(lines))
    removed += [("footer", ln["text"]) for ln in lines[anchor:]]
    return lines[:anchor]


def convert_note_pdf(path: Path):
    """Blocks of one note in reading order, the cut-paragraph log and the removed lines."""
    removed, page_lines = [], []
    with pdfplumber.open(path) as pdf:
        for page_no, page in enumerate(pdf.pages, 1):
            lines = read_lines(page, note_char_filter(page, removed))
            for ln in lines:
                ln["text"] = ln["text"].replace(PUA_BULLET, BULLET)
            page_lines.append((page_no, drop_footer(lines, removed)))
    blocks, breaks = build_blocks(page_lines, classify_note_page, {"title", "h2", "h3"})
    return blocks, breaks, removed


def render_note(blocks: list[dict], vocabulary, decisions: list, markers: list):
    """Markdown text and title of one note: ``#`` title, ``##`` sections, ``###`` sub-headings."""
    out, title = [], ""
    for block in blocks:
        text = join_lines(block["lines"], vocabulary, decisions)
        markers += NOTE_MARKER_RE.findall(text)
        text = NOTE_MARKER_RE.sub("", text)
        if block["kind"] == "title":
            title = text
            out.insert(0, f"# {text}")
        elif block["kind"] == "h2":
            out.append(f"## {text}")
        elif block["kind"] == "h3":
            out.append(f"### {text}")
        else:
            out.append("- " + text[1:].strip() if text.startswith(BULLET) else text)
    return "\n\n".join(out) + "\n", title


note_pdfs = {pe: EXTERNAL / f"eprs-ata-{pe}-{slug}.pdf" for pe, slug in BRIEFS.items()}
note_blocks, note_breaks, note_removed = {}, {}, {}
for pe, path in note_pdfs.items():
    note_blocks[pe], note_breaks[pe], note_removed[pe] = convert_note_pdf(path)
print("notes converted:", len(note_blocks))
print({pe: dict(collections.Counter(b["kind"] for b in bl)) for pe, bl in note_blocks.items()})


notes converted: 14
{'760421': {'title': 1, 'text': 8, 'h2': 3}, '751386': {'title': 1, 'text': 5, 'h2': 3}, '759620': {'title': 1, 'text': 6, 'h2': 3}, '747926': {'title': 1, 'text': 14, 'h2': 3}, '772906': {'title': 1, 'text': 9, 'h2': 3, 'h3': 1}, '779252': {'title': 1, 'text': 11, 'h2': 4}, '760355': {'title': 1, 'text': 9, 'h2': 3, 'h3': 3}, '779193': {'title': 1, 'text': 8, 'h2': 3, 'h3': 7}, '739206': {'title': 1, 'text': 10, 'h2': 6}, '739350': {'title': 1, 'text': 25, 'h2': 4}, '754642': {'title': 1, 'text': 6, 'h2': 3}, '760392': {'title': 1, 'text': 7, 'h2': 2}, '759583': {'title': 1, 'text': 6, 'h2': 3}, '760391': {'title': 1, 'text': 5, 'h2': 3}}


## Stage 3 - Write the files, read them back with the package reader

The vocabulary for split words is built from all fifteen documents, the files are written, and each file is segmented with
`split_markdown` and the `Segmenter` of the package, the code path of `DocDistance().embed_statements` without the encoder.


In [8]:
# the vocabulary of all fifteen documents decides the split words; built once, before any joining
all_blocks = report_blocks + [b for blocks in note_blocks.values() for b in blocks]
vocabulary = build_vocabulary(all_blocks)

hyphen_decisions = {"report": []}
report_md, report_heads = render_report(report_blocks, vocabulary, hyphen_decisions["report"])
(OUT_DIR / "report.md").write_text(report_md, encoding="utf-8")

note_titles, note_markers, brief_files = {}, {}, {}
for pe, slug in BRIEFS.items():
    hyphen_decisions[pe], note_markers[pe] = [], []
    md, note_titles[pe] = render_note(note_blocks[pe], vocabulary, hyphen_decisions[pe], note_markers[pe])
    brief_files[pe] = BRIEFS_DIR / f"{pe}-{slug}.md"
    brief_files[pe].write_text(md, encoding="utf-8")

all_decisions = [d for ds in hyphen_decisions.values() for d in ds]
by_reason = collections.Counter(d[3] for d in all_decisions)
print(f"written: report.md and {len(brief_files)} briefs under {OUT_DIR}")
print(f"words split at a line break: {len(all_decisions)} joins, by evidence {dict(by_reason)}")
table = Table(title="Split words decided by shape only (reason = fallback) or by a verified entry", title_style="bold cyan")
for col in ("piece 1", "piece 2", "result", "reason"):
    table.add_column(col)
for head, tail, verdict, reason in all_decisions:
    if reason != "seen":
        table.add_row(head, tail, verdict, reason)
console.print(table)


written: report.md and 14 briefs under /home/lab/workspace/learning/projects/docdistance/data/processed/relevance-fixture-eu
words split at a line break: 27 joins, by evidence {'fallback': 8, 'seen': 18, 'verified': 1}


 Split words decided by shape only (reason = 
      fallback) or by a verified entry       
┏━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┓
┃ piece 1  ┃ piece 2    ┃ result ┃ reason   ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━┩
│ national │ security   │ hyphen │ fallback │
│ supply   │ side       │ hyphen │ fallback │
│ non      │ viable     │ hyphen │ fallback │
│ cloud    │ targeted   │ hyphen │ fallback │
│ startup  │ ecosystem  │ hyphen │ fallback │
│ business │ as         │ hyphen │ fallback │
│ resource │ efficiency │ hyphen │ fallback │
│ resource │ intensive  │ hyphen │ fallback │
│ wide     │ bandgap    │ hyphen │ verified │
└──────────┴────────────┴────────┴──────────┘

In [9]:
segmenter = Segmenter()
HEADING_STATEMENT_RE = re.compile(r"^(#{1,6})\s+(.*)$")


def read_statements(path: Path) -> list[str]:
    """Statements of a markdown file as the package reader (``.md`` extension, auto) cuts them.

    The ``# `` title line is removed and the rest goes through the markdown segmentation of the
    package; no encoder is loaded and nothing is embedded.
    """
    text = _strip_titles(path.read_text(encoding="utf-8"))
    reported, _ = split_markdown(text, segmenter.split)
    return reported


def words_of(statement: str) -> int:
    """Word count of a statement without its markdown marker."""
    return len(re.sub(r"^(#{1,6}\s+|-\s+)", "", statement).split())


report_statements = read_statements(OUT_DIR / "report.md")
brief_statements = {pe: read_statements(path) for pe, path in brief_files.items()}
heading_statements = [s for s in report_statements if HEADING_STATEMENT_RE.match(s)]
assert len(heading_statements) == len(report_heads), "heading records and heading statements differ"
assert all(s == "#" * h["level"] + " " + h["text"] for s, h in zip(heading_statements, report_heads))

table = Table(title="Statements as read by the package reader (markdown, 1.1.32)", title_style="bold cyan")
table.add_column("document", style="cyan")
table.add_column("statements", justify="right")
table.add_column("headings", justify="right")
table.add_column("list items", justify="right")
table.add_column("words", justify="right")
rows = [("report", report_statements)] + [(f"{pe} {BRIEFS[pe]}", st) for pe, st in brief_statements.items()]
for name, statements in rows:
    n_heads = sum(1 for s in statements if HEADING_STATEMENT_RE.match(s))
    n_items = sum(1 for s in statements if s.startswith("- "))
    table.add_row(name, str(len(statements)), str(n_heads), str(n_items), str(sum(words_of(s) for s in statements)))
console.print(table)


               Statements as read by the package reader (markdown, 1.1.32)                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ document                                  ┃ statements ┃ headings ┃ list items ┃ words ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ report                                    │        883 │      113 │         10 │ 23192 │
│ 760421 gigabit-infrastructure-act         │         25 │        3 │          0 │   501 │
│ 751386 eu-chips-act                       │         23 │        3 │          0 │   503 │
│ 759620 european-media-freedom-act         │         29 │        3 │          0 │   539 │
│ 747926 ai-act-parliament-position         │         32 │        3 │          6 │  1161 │
│ 772906 ai-act-implementation-timeline     │         26 │        4 │          0 │   486 │
│ 779252 cybersecurity-act-review           │         51 │        4 │          0 │  1072 │
│ 760355 foreign-interference-preparedness  │         52 │        6 │          0 │  1206 │
│ 779193 simplifying-digital-laws           │         56 │       10 │          0 │  1076 │
│ 739206 european-year-of-skills            │         51 │        6 │          0 │  1241 │
│ 739350 online-age-verification            │         49 │        4 │         18 │  1288 │
│ 754642 european-health-data-space         │         29 │        3 │          0 │   563 │
│ 760392 ai-investment-indicators           │         32 │        2 │          0 │   656 │
│ 759583 strategic-technologies-platform    │         26 │        3 │          0 │   525 │
│ 760391 gdpr-cross-border-procedural-rules │         22 │        3 │          0 │   484 │
└───────────────────────────────────────────┴────────────┴──────────┴────────────┴───────┘

### Unnumbered headings and the heading tree

Unnumbered sub-headings are detected from the font record: a line in which 95 % or more of the characters are Bold 12 pt and
that carries no number. Box titles and box labels are lines of fully bold 10 pt that stand alone between two block gaps. Every
detected heading is listed below so a reader can check them against the PDF.


In [10]:
numbered = [h for h in report_heads if h["role"] == "numbered"]
table = Table(title="Detected headings that carry no number in the PDF", title_style="bold cyan")
for col in ("page", "role", "level", "heading"):
    table.add_column(col, justify="right" if col in ("page", "level") else "left")
for head in report_heads:
    if head["role"] != "numbered":
        table.add_row(str(head["page"]), head["role"], str(head["level"]), head["text"])
console.print(table)
print("headings by role:", dict(collections.Counter(h["role"] for h in report_heads)))

# numbering continuity: every n.m has its parent n, and siblings count up without a gap
numbers = [NUMBERED_HEADING_RE.match(h["text"]).group(1) for h in numbered]
seen, problems = set(), []
for number in numbers:
    parts = number.split(".")
    if len(parts) > 1 and ".".join(parts[:-1]) not in seen:
        problems.append(f"{number}: parent {'.'.join(parts[:-1])} not seen before")
    previous = ".".join(parts[:-1] + [str(int(parts[-1]) - 1)])
    if int(parts[-1]) > 1 and previous not in seen:
        problems.append(f"{number}: previous sibling {previous} not seen before")
    seen.add(number)
print(f"numbered headings: {len(numbers)}; numbering problems: {problems or 'none'}")


                                            Detected headings that carry no number in the PDF                                             
┏━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ page ┃ role       ┃ level ┃ heading                                                                                                    ┃
┡━━━━━━╇━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    4 │ chapter    │     2 │ Introduction                                                                                               │
│   11 │ boxtitle   │     5 │ Technological leadership - Recommended policies, measures and actions (49):                                │
│   11 │ boxlabel   │     6 │ Mobilising investments                                                                                     │
│   12 │ boxlabel   │     6 │ Completing the Digital Single market                                                                       │
│   12 │ boxlabel   │     6 │ Fostering cooperation between Member States                                                                │
│   18 │ boxtitle   │     6 │ Connectivity - Recommended policies, measures and actions:                                                 │
│   18 │ boxlabel   │     6 │ Completing the Digital Single market                                                                       │
│   18 │ boxlabel   │     6 │ Mobilising investments                                                                                     │
│   18 │ boxlabel   │     6 │ Fostering cooperation between Member States                                                                │
│   20 │ boxtitle   │     6 │ Semiconductors - Recommended policies, measures and actions:                                               │
│   20 │ boxlabel   │     6 │ Mobilising investments and fostering cooperation between MSs                                               │
│   21 │ boxlabel   │     6 │ Completing the Digital Single market                                                                       │
│   22 │ boxtitle   │     6 │ Edge Nodes - Recommended policies, measures and actions:                                                   │
│   22 │ boxlabel   │     6 │ Mobilising investments                                                                                     │
│   23 │ boxlabel   │     6 │ Completing the Digital Single market                                                                       │
│   24 │ boxlabel   │     6 │ 'Quantum technologies' start-ups (2023)                                                                    │
│   25 │ boxtitle   │     6 │ Quantum - Recommended policies, measures and actions:                                                      │
│   25 │ boxlabel   │     6 │ Fostering cooperation between MSs:                                                                         │
│   26 │ subheading │     6 │ Take up of cloud services                                                                                  │
│   28 │ subheading │     6 │ Take up of artificial intelligence                                                                         │
│   30 │ subheading │     6 │ Take up of big data/use of data analytics                                                                  │
│   31 │ boxtitle   │     6 │ Take up of Cloud-AI-Big Data - Recommended policies, measures and actions:                                 │
│   31 │ boxlabel   │     6 │ Mobilising investments                                                                                     │
│   31 │ boxlabel   │     6 │ Disseminating digital technologies                                                                         │
│   32 │ boxlabel   │     6 │ Completing the Digital Single market and fostering cooperation between Member 

headings by role: {'chapter': 1, 'numbered': 43, 'boxtitle': 24, 'boxlabel': 40, 'subheading': 5}
numbered headings: 43; numbering problems: none


In [11]:
NUMBER_RE = re.compile(r"^(\d+(?:\.\d+)*)\.?\s+(.*)$")


def heading_tree(statements: list[str], roles: list[str]) -> list[dict]:
    """Heading tree with statement counts below each heading, in document order.

    ``statements_own`` counts the statements between a heading and the next heading of any level,
    ``statements_total`` adds those of all deeper headings below it. Heading statements are not
    counted. ``roles`` names the kind of each heading (numbered, subheading, boxtitle ...).
    """
    nodes, stack = [], []
    for statement in statements:
        match = HEADING_STATEMENT_RE.match(statement)
        if match:
            level, text = len(match.group(1)), match.group(2)
            while stack and nodes[stack[-1]]["level"] >= level:
                stack.pop()
            number = NUMBER_RE.match(text)
            nodes.append({"index": len(nodes), "level": level, "number": number.group(1) if number else None,
                          "title": number.group(2) if number else text, "role": roles[len(nodes)],
                          "parent": stack[-1] if stack else None, "statements_own": 0, "statements_total": 0})
            stack.append(len(nodes) - 1)
        elif stack:
            for index in stack:
                nodes[index]["statements_total"] += 1
            nodes[stack[-1]]["statements_own"] += 1
    parents = {n["parent"] for n in nodes}
    for node in nodes:
        node["leaf"] = node["index"] not in parents
    return nodes


tree = heading_tree(report_statements, [h["role"] for h in report_heads])
(OUT_DIR / "sections.json").write_text(json.dumps(
    {"source": "report.md", "reader": "docdistance 1.1.32, markdown reader (auto by .md extension)",
     "statements_total": len(report_statements), "sections": tree}, indent=1, ensure_ascii=False), encoding="utf-8")

LEAF_FLOOR = 5
table = Table(title="Heading tree of the report (own = statements directly under the heading; * = leaf below "
              f"{LEAF_FLOOR})", title_style="bold cyan")
table.add_column("heading")
table.add_column("role")
table.add_column("own", justify="right")
table.add_column("total", justify="right")
for node in tree:
    label = "  " * (node["level"] - 2) + (f"{node['number']} " if node["number"] else "") + node["title"]
    flag = " *" if node["leaf"] and node["statements_own"] < LEAF_FLOOR else ""
    table.add_row(label[:100], node["role"], f"{node['statements_own']}{flag}", str(node["statements_total"]))
console.print(table)
short_leaves = [n for n in tree if n["leaf"] and n["statements_own"] < LEAF_FLOOR]
print(f"sections.json: {len(tree)} headings; leaf sections with fewer than {LEAF_FLOOR} statements: {len(short_leaves)} of "
      f"{sum(n['leaf'] for n in tree)} leaves")
print("by role among short leaves:", dict(collections.Counter(n["role"] for n in short_leaves)))


                    Heading tree of the report (own = statements directly under the heading; * = leaf below 5)                     
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━┳━━━━━━━┓
┃ heading                                                                                              ┃ role       ┃ own ┃ total ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━╇━━━━━━━┩
│ Introduction                                                                                         │ chapter    │   9 │     9 │
│ 1 Key drivers of the EU’s digital transformation in 2024                                             │ numbered   │   3 │    53 │
│   1.1 A new geopolitical paradigm                                                                    │ numbered   │   8 │     8 │
│   1.2 Fostering competitiveness in a complex economic context                                        │ numbered   │  11 │    11 │
│   1.3 Entering a new age shaped by generative AI                                                     │ numbered   │  12 │    12 │
│   1.4 Keeping people and societies on board in an increasingly hybrid context                        │ numbered   │  19 │    19 │
│ 2 A competitive, sovereign and resilient EU based on technological leadership                        │ numbered   │   1 │   320 │
│   2.1 Building digital technological leadership for future competitiveness                           │ numbered   │  18 │   182 │
│     2.1.1 Investing in research and innovation                                                       │ numbered   │  13 │    19 │
│       Technological leadership - Recommended policies, measures and actions (49):                    │ boxtitle   │   0 │     6 │
│         Mobilising investments                                                                       │ boxlabel   │ 2 * │     2 │
│         Completing the Digital Single market                                                         │ boxlabel   │ 2 * │     2 │
│         Fostering cooperation between Member States                                                  │ boxlabel   │ 2 * │     2 │
│     2.1.2 A functioning Digital Single Market as a public good for EU’s productivity                 │ numbered   │  25 │    25 │
│     2.1.3 Developing and deploying sovereign and resilient collaborative connectivity and computing  │ numbered   │   1 │   120 │
│       2.1.3.1 Gigabit connectivity infrastructure                                                    │ numbered   │  44 │    52 │
│         Connectivity - Recommended policies, measures and actions:                                   │ boxtitle   │ 0 * │     0 │
│         Completing the Digital Single market                                                         │ boxlabel   │ 1 * │     1 │
│         Mobilising investments                                                                       │ boxlabel   │   6 │     6 │
│         Fostering cooperation between Member States                                                  │ boxlabel   │ 1 * │     1 │
│       2.1.3.2 Semiconductors                                                                         │ numbered   │  31 │    34 │
│         Semiconductors - Recommended policies, measures and actions:                                 │ boxtitle   │ 0 * │     0 │
│         Mobilising investments and fostering cooperation between MSs                                 │ boxlabel   │ 1 * │     1 │
│         Completing the Digital Single market                                                         │ boxlabel   │ 2 * │     2 │
│       2.1.3.3 Edge nodes                                                                             │ numbered   │  16 │    18 │
│         Edge Nodes - Recommended policies, measures and actions:                                     │ boxtitle   │ 0 * │     0 │
│         Mobilising investments        

sections.json: 113 headings; leaf sections with fewer than 5 statements: 46 of 64 leaves
by role among short leaves: {'boxlabel': 37, 'boxtitle': 9}


## Stage 4 - Checks

Residue shows up as very short statements (broken lines, chart words) and as very long lines (paragraphs that were not split).


In [12]:
SHORT_WORDS = 4
LONG_LINE_CHARS = 2000


def short_share(statements: list[str]) -> tuple[int, int]:
    """Number of statements under ``SHORT_WORDS`` words, and how many of them are not headings."""
    short = [s for s in statements if words_of(s) < SHORT_WORDS]
    return len(short), sum(1 for s in short if not HEADING_STATEMENT_RE.match(s))


def long_lines(path: Path) -> int:
    return sum(1 for line in path.read_text(encoding="utf-8").splitlines() if len(line) > LONG_LINE_CHARS)


table = Table(title=f"Residue checks: statements under {SHORT_WORDS} words and lines over {LONG_LINE_CHARS} characters",
              title_style="bold cyan")
for col in ("document", "statements", "short", "share", "short, no heading", "lines > 2000"):
    table.add_column(col, justify="right" if col != "document" else "left")
paths = {"report": OUT_DIR / "report.md", **{pe: brief_files[pe] for pe in BRIEFS}}
statements_of = {"report": report_statements, **brief_statements}
for name, path in paths.items():
    n_short, n_prose = short_share(statements_of[name])
    n = len(statements_of[name])
    table.add_row(name if name == "report" else f"{name} {BRIEFS[name]}", str(n), str(n_short),
                  f"{n_short / n:.1%}", str(n_prose), str(long_lines(path)))
console.print(table)

kind = lambda s: "heading" if HEADING_STATEMENT_RE.match(s) else ("item" if s.startswith("- ") else "prose")
shortest = sorted(range(len(report_statements)), key=lambda i: (words_of(report_statements[i]), len(report_statements[i]), i))[:30]
table = Table(title="The 30 shortest statements of the report", title_style="bold cyan")
for col in ("#", "words", "kind", "statement"):
    table.add_column(col)
for i in shortest:
    table.add_row(str(i), str(words_of(report_statements[i])), kind(report_statements[i]), report_statements[i][:100])
console.print(table)


                   Residue checks: statements under 4 words and lines over 2000 characters                   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ document                                  ┃ statements ┃ short ┃ share ┃ short, no heading ┃ lines > 2000 ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ report                                    │        883 │    25 │  2.8% │                 1 │            1 │
│ 760421 gigabit-infrastructure-act         │         25 │     3 │ 12.0% │                 0 │            0 │
│ 751386 eu-chips-act                       │         23 │     3 │ 13.0% │                 0 │            0 │
│ 759620 european-media-freedom-act         │         29 │     3 │ 10.3% │                 0 │            0 │
│ 747926 ai-act-parliament-position         │         32 │     2 │  6.2% │                 0 │            0 │
│ 772906 ai-act-implementation-timeline     │         26 │     3 │ 11.5% │                 0 │            0 │
│ 779252 cybersecurity-act-review           │         51 │     2 │  3.9% │                 0 │            0 │
│ 760355 foreign-interference-preparedness  │         52 │     1 │  1.9% │                 0 │            1 │
│ 779193 simplifying-digital-laws           │         56 │     0 │  0.0% │                 0 │            0 │
│ 739206 european-year-of-skills            │         51 │     6 │ 11.8% │                 0 │            0 │
│ 739350 online-age-verification            │         49 │     1 │  2.0% │                 0 │            0 │
│ 754642 european-health-data-space         │         29 │     3 │ 10.3% │                 0 │            0 │
│ 760392 ai-investment-indicators           │         32 │     0 │  0.0% │                 0 │            0 │
│ 759583 strategic-technologies-platform    │         26 │     3 │ 11.5% │                 0 │            0 │
│ 760391 gdpr-cross-border-procedural-rules │         22 │     3 │ 13.6% │                 0 │            0 │
└───────────────────────────────────────────┴────────────┴───────┴───────┴───────────────────┴──────────────┘

                 The 30 shortest statements of the report                 
┏━━━━━┳━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ words ┃ kind    ┃ statement                                      ┃
┡━━━━━╇━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 462 │ 1     │ prose   │ 26                                             │
│ 0   │ 1     │ heading │ ## Introduction                                │
│ 653 │ 1     │ heading │ ###### Cooperation                             │
│ 788 │ 2     │ heading │ ##### National Roadmaps:                       │
│ 198 │ 2     │ heading │ ##### 2.1.3.2 Semiconductors                   │
│ 430 │ 2     │ heading │ ##### Mobilising investments                   │
│ 104 │ 2     │ heading │ ###### Mobilising investments                  │
│ 189 │ 2     │ heading │ ###### Mobilising investments                  │
│ 254 │ 2     │ heading │ ###### Mobilising investments                  │
│ 333 │ 2     │ heading │ ###### Mobilising investments                  │
│ 696 │ 2     │ heading │ ###### Promoting cooperation:                  │
│ 796 │ 2     │ heading │ ##### Mobilising investments:                  │
│ 857 │ 2     │ heading │ ##### Mobilising investments:                  │
│ 367 │ 2     │ heading │ ###### Mobilising investments:                 │
│ 475 │ 2     │ heading │ ###### Mobilising investments:                 │
│ 510 │ 2     │ heading │ ###### Mobilising investments:                 │
│ 537 │ 2     │ heading │ ###### Mobilising investments:                 │
│ 630 │ 2     │ heading │ ###### Mobilising investments:                 │
│ 691 │ 2     │ heading │ ###### Mobilising investments:                 │
│ 236 │ 3     │ heading │ ##### 2.1.3.3 Edge nodes                       │
│ 258 │ 3     │ heading │ ##### 2.1.3.4 Quantum computing                │
│ 400 │ 3     │ heading │ ### 2.3. Strengthening cybersecurity           │
│ 337 │ 3     │ heading │ ###### Disseminating digital technologies      │
│ 369 │ 3     │ heading │ ###### Disseminating digital technologies:     │
│ 568 │ 3     │ heading │ ###### Disseminating digital technologies:     │
│ 785 │ 4     │ heading │ ### 4.4. The way forward                       │
│ 274 │ 4     │ heading │ ###### Fostering cooperation between MSs:      │
│ 572 │ 4     │ heading │ ###### Fostering cooperation between MSs:      │
│ 598 │ 4     │ heading │ ###### Fostering cooperation between MSs:      │
│ 266 │ 4     │ heading │ ###### 'Quantum technologies' start-ups (2023) │
└─────┴───────┴─────────┴────────────────────────────────────────────────┘

### Read-through sample

25 statements of the report, drawn with a fixed seed, with their section path.


In [13]:
def section_paths(statements: list[str]) -> list[str]:
    """Heading chain of each statement; a heading statement carries its own chain."""
    stack, paths = [], []
    for statement in statements:
        match = HEADING_STATEMENT_RE.match(statement)
        if match:
            level = len(match.group(1))
            while stack and stack[-1][0] >= level:
                stack.pop()
            stack.append((level, match.group(2)))
        paths.append(" > ".join(title for _, title in stack))
    return paths


report_paths = section_paths(report_statements)
SAMPLE_SIZE = 25
sample = sorted(random.Random(SEED).sample(range(len(report_statements)), SAMPLE_SIZE))
for i in sample:
    print(f"[{i}] {report_paths[i][-110:]}\n      {report_statements[i]}\n")


[41] 1. Key drivers of the EU’s digital transformation in 2024 > 1.3. Entering a new age shaped by generative AI
      Moreover, the deployment of generative AI is expected to generate business value raging from EUR 2.4 to EUR 4.0 trillion annually.

[97] ding digital technological leadership for future competitiveness > 2.1.1. Investing in research and innovation
      To better protect the EU’s strategic assets, interests, autonomy and security, the Commission has introduced safeguards under the Horizon Europe Regulation and investment safeguards under the European Innovation Council.

[142] esilient collaborative connectivity and computing infrastructure > 2.1.3.1 Gigabit connectivity infrastructure
      Eurobarometer 2024: to facilitate their daily use of digital technologies, 4 out of 5

[143] esilient collaborative connectivity and computing infrastructure > 2.1.3.1 Gigabit connectivity infrastructure
      Europeans stress the need for better connectivity through the availabili

### Reading order and paragraph joins

Reading order is checked on pages with boxes or figures beside body text against the page text of the PDF; the joins of cut
paragraphs are listed so each can be judged.


In [14]:
def sequence_check(pdf, pages: list[int], blocks: list[dict]) -> list[tuple]:
    """For pages with boxes or figures: do the block openings follow the order of the page text?

    The page text of pdfplumber (all characters, top to bottom) is the reference; each block of
    the page must start at a later position than the block before it.
    """
    rows = []
    for page_no in pages:
        reference = " ".join((pdf.pages[page_no - 1].extract_text() or "").split())
        position, in_order, total = 0, 0, 0
        for block in blocks:
            if block["page"] != page_no:
                continue
            opening = " ".join(block["lines"][0].split()[:6])
            found = reference.find(opening, position)
            total += 1
            if found >= 0:
                in_order, position = in_order + 1, found
        rows.append((page_no, total, in_order))
    return rows


# pages with a recommendation box, a text box, a green quote box, or a figure beside body text
ORDER_PAGES = [11, 12, 14, 16, 18, 19, 26, 28, 30, 39, 53, 62, 66]
with pdfplumber.open(REPORT_PDF) as pdf:
    rows = sequence_check(pdf, ORDER_PAGES, report_blocks)
table = Table(title="Reading order against the page text of the PDF (blocks whose opening words follow in order)",
              title_style="bold cyan")
for col in ("page", "blocks", "in order"):
    table.add_column(col, justify="right")
for page_no, total, in_order in rows:
    table.add_row(str(page_no), str(total), str(in_order))
console.print(table)

with pdfplumber.open(REPORT_PDF) as pdf:
    last_text_pages = [p for p in range(REPORT_FIRST_PAGE, REPORT_LAST_PAGE + 1) if not any(b["page"] == p for b in report_blocks)]
print("pages inside the content range that produced no block (full-page figures):", last_text_pages)
print("cut paragraphs: joined", sum(1 for b in report_breaks if b[2]), "kept apart", sum(1 for b in report_breaks if not b[2]),
      "- of the joined ones,", sum(1 for b in report_breaks if b[2] and b[1] == "page"), "span a page break")
print("joins across a figure or box on the same page (the next piece starts with a lower-case letter):")
for b in report_breaks:
    if b[2] and b[1] == "gap":
        print(f"  p{b[0]} gap joined: ...{b[3]} | {b[4]}...")
print("page breaks joined although the next piece starts with a capital (decided by the missing full stop only):")
for b in report_breaks:
    if b[1] == "page" and b[2] and not b[4][:1].islower():
        print(f"  p{b[0]} page joined: ...{b[3]} | {b[4]}...")


 Reading order against the  
page text of the PDF (blocks
 whose opening words follow 
         in order)          
┏━━━━━━┳━━━━━━━━┳━━━━━━━━━━┓
┃ page ┃ blocks ┃ in order ┃
┡━━━━━━╇━━━━━━━━╇━━━━━━━━━━┩
│   11 │      5 │        5 │
│   12 │     11 │       11 │
│   14 │      4 │        4 │
│   16 │      7 │        7 │
│   18 │     13 │       13 │
│   19 │      5 │        5 │
│   26 │      8 │        8 │
│   28 │      4 │        4 │
│   30 │      4 │        4 │
│   39 │      7 │        7 │
│   53 │      9 │        9 │
│   62 │      8 │        8 │
│   66 │      7 │        7 │
└──────┴────────┴──────────┘

pages inside the content range that produced no block (full-page figures): [48, 75]
cut paragraphs: joined 13 kept apart 184 - of the joined ones, 13 span a page break
joins across a figure or box on the same page (the next piece starts with a lower-case letter):
page breaks joined although the next piece starts with a capital (decided by the missing full stop only):
  p20 page joined: ... the global market by value according to | International Data Corporation estimatio...
  p60 page joined: ...les, the Commission Recommendation of 12 | December 2023 asked Member States to tak...
  p72 page joined: ...on - including pre-commercial deployment | - for innovative digital technologies. T...


### Word accounting

Words of each PDF against the words of its markdown file, by what was removed. The categories are read from the PDF with the
same style tests the conversion uses.


In [15]:
def words_in(page, predicate) -> int:
    """Word count of the characters of a page that satisfy ``predicate`` (read in page order)."""
    kept = page.filter(lambda o: o["object_type"] != "char" or predicate(o))
    return len((kept.extract_text() or "").split())


def md_words(path: Path) -> int:
    """Words of a markdown file without heading hashes and list markers."""
    return sum(len(re.sub(r"^(#{1,6}\s+|-\s+)", "", line).split()) for line in path.read_text(encoding="utf-8").splitlines())


def report_accounting(path: Path) -> dict[str, int]:
    """Words of the annex PDF by what happened to them."""
    categories = {
        "figure captions (Medium Italic 11)": lambda c: char_style(c) == ("MediumItalic", 11.0),
        "footnote text (Italic 8)": lambda c: char_style(c)[0] == "Italic" and 7 <= char_style(c)[1] < 9.5,
        "footnote markers (Italic under 7)": lambda c: char_style(c)[0] == "Italic" and char_style(c)[1] < 7,
        "page numbers (Medium 8)": lambda c: char_style(c)[0] == "Medium" and char_style(c)[1] <= 8.0,
    }
    chart = lambda c: not is_report_text(c) and not any(f(c) for f in categories.values())
    counts = collections.Counter()
    with pdfplumber.open(path) as pdf:
        counts["pages outside 4 to 75 (cover, legal notice, blank, imprint)"] = sum(
            len((pdf.pages[i - 1].extract_text() or "").split())
            for i in range(1, len(pdf.pages) + 1) if not REPORT_FIRST_PAGE <= i <= REPORT_LAST_PAGE)
        counts["pdf total"] = sum(len((p.extract_text() or "").split()) for p in pdf.pages)
        counts["kept text (words in the line blocks)"] = sum(len(" ".join(b["lines"]).split()) for b in report_blocks)
        for i in range(REPORT_FIRST_PAGE, REPORT_LAST_PAGE + 1):
            page = pdf.pages[i - 1]
            for name, predicate in categories.items():
                counts[name] += words_in(page, predicate)
            counts["chart and other small text"] += words_in(page, chart)
    return counts


counts = report_accounting(REPORT_PDF)
output_words = md_words(OUT_DIR / "report.md") - len(REPORT_TITLE.split())
kept_words = counts["kept text (words in the line blocks)"]
table = Table(title="Annex: words of the PDF text against the words of report.md", title_style="bold cyan")
table.add_column("what")
table.add_column("words", justify="right")
for name, value in counts.items():
    table.add_row(name, f"{value:,}")
table.add_row("report.md (without title, heading hashes and list markers)", f"{output_words:,}")
table.add_row("kept text minus report.md (joins of split words, bullet signs)", f"{kept_words - output_words:,}")
parts = sum(v for k, v in counts.items() if k != "pdf total")
table.add_row("pdf total minus the sum of the parts (words cut or merged at part borders)", f"{counts['pdf total'] - parts:,}")
console.print(table)
print("joins of split words in the annex:", len(hyphen_decisions["report"]), "; bullet signs:",
      sum(1 for b in report_blocks if b["lines"][0].startswith(BULLET)))


              Annex: words of the PDF text against the words of report.md              
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ what                                                                       ┃  words ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ pages outside 4 to 75 (cover, legal notice, blank, imprint)                │    136 │
│ pdf total                                                                  │ 28,740 │
│ kept text (words in the line blocks)                                       │ 23,225 │
│ figure captions (Medium Italic 11)                                         │    286 │
│ footnote text (Italic 8)                                                   │  3,875 │
│ footnote markers (Italic under 7)                                          │    170 │
│ page numbers (Medium 8)                                                    │    153 │
│ chart and other small text                                                 │    897 │
│ report.md (without title, heading hashes and list markers)                 │ 23,192 │
│ kept text minus report.md (joins of split words, bullet signs)             │     33 │
│ pdf total minus the sum of the parts (words cut or merged at part borders) │     -2 │
└────────────────────────────────────────────────────────────────────────────┴────────┘

joins of split words in the annex: 20 ; bullet signs: 10


In [16]:
def note_accounting(pe: str) -> dict[str, int]:
    """Words of one note PDF by what happened to them."""
    removed_words = collections.Counter()
    for reason, text in note_removed[pe]:
        removed_words[reason] += len(text.split())
    size = lambda c: char_style(c)[1]
    counts = collections.Counter()
    with pdfplumber.open(note_pdfs[pe]) as pdf:
        counts["pdf total"] = sum(len((p.extract_text() or "").split()) for p in pdf.pages)
        for page in pdf.pages:
            counts["labels"] += words_in(page, lambda c: 15.5 <= size(c) <= 20.5)
            counts["captions"] += words_in(page, lambda c: size(c) == 12.0 and char_style(c)[0] != "Bold")
            counts["9 pt text"] += words_in(
                page, lambda c: 8.0 < size(c) < NOTE_BODY_SIZE_RANGE[0] and not in_range(size(c), SUPERSCRIPT_SIZE_RANGE))
            counts["disclaimer"] += words_in(
                page, lambda c: size(c) <= 8.0 and c["top"] > 700)
    counts["footer"] = removed_words["footer"]
    counts["source"] = removed_words["source line"]
    counts["kept"] = sum(len(" ".join(b["lines"]).split()) for b in note_blocks[pe])
    return counts


table = Table(title="Notes: words of the PDF text against the words of the markdown file", title_style="bold cyan")
columns = ("pdf total", "labels", "captions", "9 pt text", "disclaimer", "footer", "source", "kept")
table.add_column("note")
for col in columns + ("md", "unexplained"):
    table.add_column(col, justify="right")
for pe in BRIEFS:
    c = note_accounting(pe)
    unexplained = c["pdf total"] - sum(c[k] for k in columns[1:])
    table.add_row(pe, *[str(c[k]) for k in columns], str(md_words(brief_files[pe])), str(unexplained))
console.print(table)
print("labels = series labels, running header and the EN tag; footer = EPRS line, author, reference, date;")
print("source = figure source lines; kept = words in the line blocks; md = words of the markdown file (title included, hashes and list markers excluded).")


                       Notes: words of the PDF text against the words of the markdown file                       
┏━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━┳━━━━━━━━━━━━━┓
┃ note   ┃ pdf total ┃ labels ┃ captions ┃ 9 pt text ┃ disclaimer ┃ footer ┃ source ┃ kept ┃   md ┃ unexplained ┃
┡━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━╇━━━━━━━━━━━━━┩
│ 760421 │       624 │      9 │        0 │         0 │         92 │     18 │      0 │  505 │  504 │           0 │
│ 751386 │       623 │      8 │        0 │         0 │         92 │     17 │      0 │  506 │  506 │           0 │
│ 759620 │       661 │      8 │        0 │         0 │         92 │     17 │      0 │  544 │  543 │           0 │
│ 747926 │      1303 │     17 │        0 │         0 │         92 │     17 │      0 │ 1177 │ 1169 │           0 │
│ 772906 │       633 │     13 │       17 │         0 │         92 │     20 │      0 │  491 │  490 │           0 │
│ 779252 │      1224 │     15 │        0 │         0 │        114 │     17 │      0 │ 1078 │ 1078 │           0 │
│ 760355 │      1334 │     14 │        0 │         0 │         92 │     17 │      0 │ 1211 │ 1211 │           0 │
│ 779193 │      1206 │     15 │        0 │         0 │         92 │     17 │      0 │ 1082 │ 1082 │           0 │
│ 739206 │      1381 │     13 │        2 │         4 │         92 │     20 │      0 │ 1250 │ 1249 │           0 │
│ 739350 │      1436 │     15 │        0 │         0 │         92 │     17 │      0 │ 1312 │ 1294 │           0 │
│ 754642 │       685 │      8 │        0 │         0 │         92 │     18 │      0 │  567 │  567 │           0 │
│ 760392 │      1021 │     15 │       37 │       171 │         92 │     23 │     20 │  663 │  662 │           0 │
│ 759583 │       651 │      9 │        0 │         0 │         92 │     20 │      0 │  530 │  530 │           0 │
│ 760391 │       612 │      9 │        0 │         0 │         92 │     17 │      0 │  494 │  494 │           0 │
└────────┴───────────┴────────┴──────────┴───────────┴────────────┴────────┴────────┴──────┴──────┴─────────────┘

labels = series labels, running header and the EN tag; footer = EPRS line, author, reference, date;
source = figure source lines; kept = words in the line blocks; md = words of the markdown file (title included, hashes and list markers excluded).


In [17]:
readme = f"""# relevance-fixture-eu - one long EU report and 14 short briefing notes

Text fixture for relevance work on documents of different length: one long report and 14 short notes on
related digital-policy subjects, converted from public PDFs to clean markdown. Nothing here is scored; no
note is labelled against the report. Labels are assigned elsewhere.

## Contents

- `report.md` - State of the Digital Decade 2024, Annex 1 (European Commission): {len(report_statements)} statements as read by the package reader, {len(report_heads)} headings
- `briefs/<PE>-<slug>.md` - 14 EPRS "At a glance" notes (European Parliamentary Research Service), {min(len(s) for s in brief_statements.values())} to {max(len(s) for s in brief_statements.values())} statements each
- `sections.json` - heading tree of the report: level, number, title, role, parent, and the count of statements below each heading (`statements_own` directly below, `statements_total` with the deeper headings)

Provenance, titles, source URLs and reuse notices of the 15 documents: `data/external/eu-digital-decade-relevance_origin.md`.

## Format

- first line `# <title>`, which the markdown reader removes; one paragraph per line, never a wrapped paragraph
- headings follow the reader's rules: one statement per heading, list item (`- `) and paragraph sentence
- report heading level follows the numbering depth (`##` for `1.`, `###` for `1.1.`, `####` for `2.1.3.`); unnumbered
  sub-headings, recommendation box titles and box labels sit below their numbered parent (never past six hashes)
- notes: `##` section headings (14 pt in the PDF), `###` bold sub-headings and box titles

## How it was built

Notebook `notebooks/14-kj-eu-relevance-fixture.ipynb` reads the PDFs in `data/external/`, keeps text by font and size
with `pdfplumber`, joins wrapped lines into paragraphs, repairs words split at a line break, and writes these files.
Statements are counted with the markdown segmentation of the package (`docdistance` 1.1.32), without an encoder.

## Removed

- report: cover, legal notice, blank page and imprint pages, page numbers, footnote text and markers, chart labels, figure captions
- notes: series labels, running header, EPRS footer with author, reference and date, disclaimer and copyright line, chart,
  table and infographic text with captions, figure source lines, footnote markers (`*`)

## Limits

- the text is not edited; typing errors of the sources stay (for example `alҐso`, `four in10`, `74°%` and `1025` for 10 to the power 25)
- paragraph, heading and box detection rely on font size and line spacing; see the notebook for the pages where it can fail
- the annex PDF has no table of contents (pages 3 and 75 are text-free decoration); its recommendation boxes become a box title and box labels, so {len(short_leaves)} of {sum(n['leaf'] for n in tree)} leaf sections hold fewer than {LEAF_FLOOR} statements; all of them have the role `boxtitle` or `boxlabel` in `sections.json`, none is a numbered section or a sub-heading
- box titles and labels under a five-level numbered heading share level six, so the box title is an empty leaf there
- the notes keep the closing box of procedure references (first-reading report, rapporteur) where the PDF has one
- the PDFs are public documents kept for calibration and fixtures only; they are not redistributed beyond the licence and not used for training
"""
(OUT_DIR / "README.md").write_text(readme, encoding="utf-8")
print(f"README.md written ({len(readme.splitlines())} lines)")


README.md written (42 lines)


## Conclusions

**Result.** `data/processed/relevance-fixture-eu/` holds `report.md` (883 statements, 113 headings, 23,192 words), 14 notes in `briefs/` (22 to 56 statements each, 484 to 1,288 words), `sections.json` and `README.md`. Nothing is scored or labelled.

**Residue checks.**

- Statements under 4 words: 25 of 883 in the report (2.8 %); 24 are headings of one to three words and one is the prose fragment `26`, which the sentence segmenter cut from `26 Member States provided ...` (the line in `report.md` is intact). No note has a short statement outside its headings (the share of short statements in a note, 0 to 14 %, is its own headings).
- Lines over 2000 characters: 2. Both are single paragraphs as typeset: the Digital Services Act box of the annex (2,437 characters, continuous line spacing over two pages) and the second long paragraph of note 760355 (about 2,500 characters, seen in the page image).
- Word accounting: the annex reconciles to 2 words (cover, legal notice, imprint, footnotes, markers, captions, chart text, page numbers); all 14 notes reconcile to 0 words. The 33 words between the kept lines and `report.md` are 10 bullet signs, 20 joins of split words and 3 joins at a hyphen before a capital or digit (`HPC-QS`, `16-74`, `Israeli-Palestinian`). Pages 48 (full-page figure) and 75 (blank) produce no text.
- Reading order: on 13 annex pages with boxes or figures, every block opens at a later position of the PDF page text than the block before it. Looked at as page images against the output: pages 3, 11, 12, 26, 39 and 48 singly, pages 14 to 21 and 30 to 37 as contact sheets, and page 1 of note 760355.

**Headings.** 43 numbered headings with no gap in the numbering, 1 unnumbered chapter title (`Introduction`), 5 unnumbered section headings (Bold 12 pt, listed above), 24 box titles and 40 box labels. 46 of 64 leaf sections hold fewer than 5 statements; every one is a box title or a box label, none is a numbered section or a sub-heading. Filter on `role` in `sections.json` to drop them.

**Heuristic steps and where they fail.**

- Text is kept by font and size. Annex: upright or italic text of 9.5 pt and above; the only text of that size dropped is 28 caption lines and one chart title (`AI Venture Capital investments (2022)`, page 29). Notes: 10.0 to 11.3 pt, 14 pt section headings, bold 12 pt sub-headings, 24 pt titles. A body text set below 9.5 pt (annex) or 10 pt (notes) would be lost; none was found by the word accounting.
- Headings come from whole-line bold at 24 pt and 12 pt (annex) and 14 pt regular or bold 12 pt (notes). Unnumbered sub-headings are separated reliably by the font record; the box titles and labels (bold 10 pt run standing alone between two gaps, at most 120 characters, no final full stop) are the least certain step: a bold lead-in that stands alone on its line would be taken as a label.
- Paragraph breaks use a gap larger than 1.12 times the median line pitch of the font size, measured per document (pitch 1.14 to 1.40 times the size, so a fixed gap fails: the paragraph gap of note 760421 is only 1.5 times the size). Paragraphs that are separated by less than that gap merge: the two closing lines of the box of note 760421 are one paragraph, in the other notes two.
- A paragraph cut by a page break joins when the next piece starts lower-case or the cut piece has no closing full stop (13 page joins, 3 of them before a capital: pages 20, 60, 72, all continuing sentences); a cut on the same page (figure or box) joins only before a lower-case start (0 cases). A paragraph that ends without full stop at a page end and is followed by a new paragraph would be joined wrongly.
- Words split at a line break: 27 joins; 18 decided by a form seen elsewhere in the 15 documents, 8 by the shape of the two pieces (all kept their hyphen) and 1 entry (`wide-bandgap`) set after checking the PDF. No soft-hyphenated word had to be glued. A split word whose glued form is not in the vocabulary and whose pieces are not words would be glued.
- Footnote markers are removed by style (Italic under 7 pt) in the annex and by the pattern `*` after a word in the notes (2 markers); no `(n)` pattern is left in the annex. A real superscript would be removed with them; the one superscript of the notes (`10` to the power `25`) is kept as the digits `1025`.
- Figure source lines of the notes are removed by the prefix `Source:` at 10 pt (4 lines); a box paragraph that starts with `Source:` would be removed too.
- Notes: the footer is cut at the line `EPRS | European Parliamentary Research Service` and everything below it; the closing box of procedure references stays as a paragraph. Lists exist in two notes (Wingdings arrow, 24 items, written as `- `) and in the annex (bullet sign, 10 items); items that carry only a bold lead-in and no marker in the PDF (note 760421) are paragraphs.

**Not verified.** The 15 source URLs were not requested (no network). Text of the PDFs is not compared word by word with a second extraction tool. Of the annex, the opening and closing words of every paragraph were read, and the full text of pages 4 to 6, 55 and 66 (line dump) and of the 25-statement sample; the rest was checked by the statistics above. The source typing errors listed in `README.md` were left unedited.
